# Задание 2 (файнтюн моделей)

Дедлайн:  3 мая, 23:59

В этом задании вам предлагается дообучить предобученную языковую модель с использованием LoRA.

Для этого задания вам нужна GPU, но Free Tier T4 должно хватить, чтобы выполнить все задания.

В качестве базовой модели предлагается взять `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (1.1B). Вы также можете взять `microsoft/phi-2`. Обе модели нужно будет загрузить в 4-битном формате (NF4 через `bitsandbytes`)

Основным датасетом будет служить SNLI, который содержит некоторую предпосылку и гипотезу, а задача модели выдать по этой паре один из трех лейблов:  entailment (следование), contradiction (противоречие), and neutral (нет связи между предпосылкой и следствием). Этот датасет - классический для задачи Natural Language Inference. Возьмите 10%-подвыборка со стратификацией (~55k train, ~1k val, ~1k test). Зафиксируйте random seed.

Ссылка: https://nlp.stanford.edu/projects/snli/

Внимание! Полные баллы за задачу будут выставлены только при наличии анализа и обсуждения полученных результатов.

## Часть 1 — LoRA Fine-Tuning для NLI (20 баллов)

Дообучите базовую модель на подвыборке SNLI с помощью LoRA-адаптеров через библиотеку `peft`. Сформулируйте NLI как задачу генерации текста: разработайте шаблон промпта-инструкции 

(например, `Premise: ... Hypothesis: ... Label:`)

 и обучите модель генерировать токен метки. В отчёте обсудите, почему для decoder-only моделей необходима генеративная постановка и какие риски она создаёт (например, генерация непредвиденных токенов на инференсе — напишит как с этим можно справиться).

Выберите гиперпараметры LoRA (`r, alpha, target_modules, dropout`). Задокументируйте и обоснуйте каждый выбор. Используйте r=16 по умолчанию, либо если же вы определите другие оптимальные значения, используйте их.

Учите в течение 2–3 эпох с размером батча, помещающимся на GPU (при необходимости используйте `gradient accumulation`). Отобразите training loss и validation accuracy по эпохам

Проведите исследование по параметру рангу LoRA $r ∈ {4, 16, 64}$. Для каждого значения укажите количество обучаемых параметров vs. общее число параметров (в процентах и абсолютных числах) и приведите accuracy на валидационной выборке.

Сравните лучшую QLoRA-модель с двумя бейзлайнами:

* Zero-shot промптинг: используйте тот же шаблон промпта, но без дообучения. Попробуйте два варианта: промпт как он есть и промпт с 3 вручную написанными few-shot примерами.
*  Linear probe: извлеките hidden state последнего токена из замороженной модели и обучите только линейный 3-классовый классификатор поверх него. Используйте `torch.no_grad()` при извлечении признаков для экономии памяти.

Приведите accuracy и время обучения для всех подходов. В отчёте проанализируйте, при каких условиях и бюджетах вычислений каждая стратегия предпочтительна.

## Часть 2 — Анализ квантизации (10 баллов)

Вы уже обучили модель в 4-бит с QLoRA. Теперь сравните качество инференса при разных точностях:


* Загрузите базовую модель + объединённый LoRA-адаптер в 16-бит (если помещается — если получите OOM, задокументируйте ошибку и оцените, какой GPU потребуется исходя из размера модели).
* Загрузите в 8-бит (`load_in_8bit=True`).
* Загрузите в 4-бит (ваша текущая конфигурация).

Для каждого варианта приведите: accuracy на валидационной выборке, скорость инференса (примеров/сек на 200 валидационных примерах) и объём занимаемой GPU-памяти (`torch.cuda.memory_allocated` после загрузки). 

## Часть 3 — Дистилляция знаний (15 баллов)

Используйте вашу дообученную QLoRA TinyLlama (1.1B) в качестве учителя, а `google/flan-t5-small` (80M параметров) или другую модель похожего размера — в качестве ученика. Реализуйте дистилляцию на основе KL-дивергенции:

Прогоните всю обучающую подвыборку через учителя и закешируйте его softmax-распределения (с температурой $τ ∈ \{1, 3, 5\})$ по 3 классам NLI.

Обучите ученика с взвешенной комбинацией hard-label cross-entropy и soft-label KL loss:
$$ L = α · L_\text{soft}  +  (1 - α) · L_\text{hard} $$


Замечание: внимательно посмотрите на лосс применяемый при обучении, не забудьте поделить на $τ$ оба набора логитов и домножить потом KL-loss обратно на $\tau^2$.

In [ ]:
import torch.nn.functional as F

# напоминание
soft_loss = F.kl_div(
        soft_student,
        soft_teacher,
        reduction="batchmean",
) * (tau ** 2)

### Важное замечание о несовпадении словарей:

У моделей `TinyLlama` и `flan-t5` используются разные токенизаторы, поэтому вычислить KL-дивергенцию по полным распределениям словаря невозможно. Вместо этого для каждой модели:


1. Выберите для каждого класса однотокенное слово-метку (проверьте с помощью tokenizer.encode). Например:
    ```
    LABEL_WORDS = {"entailment": "entail", "neutral": "neutral", "contradiction": "contra"}
    ```
    Обращайте внимание на порядок лейблов.

2. Обучите учителя предсказывать новые метки. Извлекайте логиты только по трем позициям, соответствующих токенам классов. Получится массив размером (N, 3).

3. Вычислите KL между 3-классовыми распределениями учителя и ученика.


Закешируйте распределение учителя (N, 3) один раз перед обучением, чтобы не перебоучать его на каждой эпохе.


 Поэкспериментируйте как минимум с двумя коэффициентами смешивания ($α ∈ \{0.3, 0.7\}$ или на ваш выбор).

Сравните все 6 пар коэффициентов (3 значения τ × 2 значения α) по accuracy ученика на валидационной выборке. Сравните с accuracy учителя

## Часть 4 - Состязательная выборка (10 баллов)

Для оценки NLI-моделей часто используются так называемые состязательные выборки, нацеленные на конкретные режимы отказа модели. Мы собрали для вас датасет из 27 примеров на три вида отказа моделей,:

* Опечатки
* Чувствительность к отрицанию: пары, в которых добавление или удаление отрицания меняет метку.
* Ловушки лексического пересечения + убеждение модели через фразу true is true.

Датасет лежит в репозитории,  `hw2_nli.csv`

Оцените все ваши модели (zero-shot, few-shot, linear probe, QLoRA, дистиллированный ученик) на вашем состязательном наборе. Представьте таблицу accuracy по феноменам. Какой получился наиболее слабый феномен для вашей лучшей модели? Добавьте несколько примеров

## Бонус — Анализ паттернов внимания (15 баллов)

Возьмите вашу лучшую QLoRA-модель и извлеките attention maps для 10 примеров NLI (смесь правильных и неправильных предсказаний). Это можно сделать с помощью хуков на слоях внимания или через прямой выход модели.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
    output_attentions=True,  # <-- для получения хуков
)

### подготовка данных, в inputs лежит токенизированный промпт

# Cпособ 1
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# получаем
# outputs.attentions — кортеж из тензоров, по одному на слой
# Каждый тензор имеет форму (batch_size, num_heads, seq_len, seq_len)

In [ ]:
# Способ 2 (хуки)
# Полезен, когда output_attentions недоступен или когда нужно
# перехватить промежуточные значения (например, Q, K, V отдельно)

attention_maps = {}
def make_hook(layer_name):
    def hook_fn(module, input, output):
        # Для TinyLlama (архитектура LlamaAttention):
        # output — это кортеж (attn_output, attn_weights, past_key_value)
        # attn_weights имеет форму (batch, num_heads, seq_len, seq_len)
        #
        # важно! структура output зависит от архитектуры модели!
        # Проверьте для вашей модели, напечатав type(output) и len(output)
        if isinstance(output, tuple) and len(output) >= 2:
            attn_weights = output[1]  # attention weights
            if attn_weights is not None:
                attention_maps[layer_name] = attn_weights.detach().cpu()
    return hook_fn

hooks = []
layers_to_hook = [0, 10, 21]  

for layer_idx in layers_to_hook:
    layer = model.model.layers[layer_idx].self_attn
    hook = layer.register_forward_hook(make_hook(f"layer_{layer_idx}"))
    hooks.append(hook)

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

for hook in hooks:
    hook.remove()

Замечание: Способ через output_attentions=True проще и надёжнее, однако способ с хуками может быть полезен, если нужно будет анализировать QKV-векторы. Вы можете выбрать любой из способов для этого задания.

Визуализируйте головы внимания для 2–3 слоёв (ранний, средний, поздний) в виде тепловых карт (heatmaps) по входным токенам.

Попробуйте найти хотя бы одну голову, которая фокусируется на границе между посылкой и гипотезой или одну с широким «усредняющим» паттерном. Если вы не нашли таких голов, предположите на каких слоях они могут быть.


Как эти паттерны соотносятся с теоретическими ролями различных голов в multi-head attention (MHA)? Кратко объясните, как GQA (Grouped Query Attention) и MLA (Multi-head Latent Attention) модифицируют стандартный MHA и какие компромиссы они вносят. Полезные статьи: GQA: [Ainslie et al. 2023,](https://arxiv.org/abs/2305.13245) MLA: [DeepSeek-V2](https://arxiv.org/abs/2405.04434)